# Checkerboard data analysis

Contact: laquitainesteeve@gmail.com based on Guilhelm's guilhelm-dev branch  

Description: This notebook is the second one of the Analysis Pipeline. If you haven't run the first one (Preprocessing) until the extraction of phy's data , you don't have the files needed to run this one. In this notebook, you will use phy sorted cluster spikes of a checkerboard recording to compute clusters raster plots, ganglion cells STA and fit ellipses on their receptive fields.

Execution time: <15 min

Tested on Ubuntu 24.04.1 LTS (32 cores, 188 GB RAM, Intel(R) Core(TM) i9-14900K ＠3.2 GHz/5.8 GHz)

**Requirements:**
- Run preprocessing before

**Required by**:
- 4-Chirp+Cell Typing.ipynb notebook

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2

# import packages
import os
import utils

# import custom packages
import params
from utils import analyse_checkerboard as analysis

In [ ]:
# --- Backup reminder: running the analysis pipeline (Step 6 of the backup guide) ---
utils.backup_reminder(
    "Step 6 - Running the analysis pipeline",
    [
        "You can adapt the analysis to your needs.",
        "At the end you will back up BOTH the Analysis output folder and the Pipeline code folder (Step 7).",
    ],
)

## Load data

- All the variables used in this part of the cell should always refer to your 'params.py' file, unless you want to manually change them only for this run (i.e. debugging). You may have to add those variable into the function you want to adapt as only the minimal amount of var are currently given to functions as inputs.
- Mandatory for each run !
- Execution time: 9 secs

In [ ]:
%%time 

# === Choose the stimulus type ===
# False = checkerboard  |  True = SWN (Shifting White Noise). SWN file paths are set in params.py.
is_swn = True

# ask user to input the parameters and create the analysis directory
# (parameters are normally in the recording name; for SWN only the recording and
#  stimulus frequency are asked \u2014 the check count/size questions are skipped)
(
    recording_number,
    recording_name,
    stimulus_frequency,
    nb_checks_x,
    nb_checks_y,
    nb_pixels_per_check,
    check_directory,
) = analysis.get_all_inputs_for_checkerboard_analysis(params, is_swn=is_swn)

if is_swn:
    # SWN: reconstruct the stimulus from its .bin/.vec and also get the stimulus
    # covariance C_I used to decorrelate (whiten) the STA further down.
    checkerboard_spikes, triggers, nb_repeats, cells_id, checkerboard, C_I = (
        analysis.load_swn_data(params, recording_name, stimulus_frequency)
    )
    # one STA pixel spans 'shift' DMD pixels (the spatial down-sampling used for SWN)
    nb_pixels_per_check = params.swn_shift_x
else:
    # Checkerboard: load or regenerate the checkerboard stimulus (no decorrelation needed).
    C_I = None
    checkerboard_spikes, triggers, nb_repeats, cells_id, checkerboard = (
        analysis.load_checkerboard_data(
            params,
            check_directory,
            recording_name,
            nb_checks_x,
            nb_checks_y,
            stimulus_frequency,
        )
    )

# rep / non-rep sequence portions (checkerboard only; SWN has no repeated sequence)
repeated_sequence_portion = (
    0.5,
    1,
)  # portion holding the repeated sequence (second half)
non_repeated_sequence_portion = (
    0,
    0.5,
)  # portion holding the random sequence (first half)

## Extract responses to repeated sequence (raster plot and psth)

- Analysis on repeated sequence: all the variables used in this part of the cell should always refere to your 'params.py' file
unless you want to manually change them only for this run (i.e. debugging). 
You may have to add those variable into the function you want to adapt as only the minimal 
amount of var are currently given to functions as inputs.
- Requirements: cell 1 run
- Execution time: < 3 mins

In [ ]:
%%time

if is_swn:
    # SWN has no repeated sequence, so there are no reliability rasters to extract.
    print(
        "SWN stimulus has no repeated sequence \u2014 skipping the reliability rasters."
    )
    rep_seq_data = None
else:
    # inputs
    save_data = True  # save data or not
    show_all_rasters = (
        True  # show all rasters in one figure in console to check extraction
    )
    save_single_cell_figures = (
        False  # compute and save one figure per cell (raster+psth); can be skipped
    )

    # extract data
    rep_seq_data = analysis.extract_all_cell_responses_to_repeated_sequences(
        checkerboard_spikes,
        triggers,
        nb_repeats,
        stimulus_frequency,
        cells_id,
        nb_frames_per_sequence=params.nb_frames_by_sequence,
        sequence_portion=repeated_sequence_portion,
    )
    if save_data:
        utils.save_obj(
            rep_seq_data, os.path.join(check_directory, "Check_rasters_data")
        )

    # plot data
    if show_all_rasters:
        analysis.plot_all_rasters(rep_seq_data, cells_id)
    if save_single_cell_figures:
        analysis.plot_and_save_single_cell_rasters(
            rep_seq_data,
            cells_id,
            check_directory,
            title="Response to repeated sequence",
        )

## Compute STAs

- **Requirements**: requires cell 1 run
- Execution time: < 5 min

In [ ]:
%%time 

# Extract responses to non-repeated sequences, compute STA and save data
sta_data_file, sta_data = analysis.compute_spike_triggered_average(
    checkerboard_spikes,
    checkerboard,
    triggers,
    nb_repeats,
    stimulus_frequency,
    check_directory,
    temporal_dimension=params.sta_temporal_dimension,
    sequence_portion=non_repeated_sequence_portion,
    nb_frames_per_sequence=params.nb_frames_by_sequence,
    sta_data_filename="sta_data_3D.pkl",
)

In [ ]:
%%time 

# Check STA computation for one example cell
# ask the user to input a cell and plot its 3D sta (e.g., 1)
analysis.plot_one_cell_3D_spike_triggered_average(
    sta_data,
    max_frames_to_show=params.sta_temporal_dimension,
    n_frames_per_line=10,
    fontsize=14,
)

## Analyse STAs to identify receptive fields

- **Requirements**: requires cell 1 run and cell 4 output
- Execution time: 10 secs

In [ ]:
%%time 

load_sta_data = True  # load previously computed sta data or not (if not, it will compute it again and overwrite the previous one if it exists)
show_all_stas = True
get_extended_analysis = True

if load_sta_data:
    sta_data_file = os.path.join(check_directory, "sta_data_3D.pkl")
    sta_data = utils.load_obj(sta_data_file)

# Analyze STAs to extract receptive field properties for all cells,
# add it to the sta_data dictionary and save it
sta_data_analysed = analysis.analyse_all_stas(
    sta_data,
    directory=check_directory,
    data_filename="sta_data_analysed.pkl",
    method="standard",
)

# SWN only: the shifting-noise stimulus is spatially correlated, so whiten each spatial
# STA by the inverse stimulus covariance (C_I) and re-fit the ellipse. The checkerboard
# is already white and skips this. It overwrites the RF in place, so the steps below are unchanged.
if is_swn:
    sta_data_analysed = analysis.decorrelate_spatial_stas(sta_data_analysed, C_I)

# Extend analysis to physical units
# adding to the analysis dictionary the RF properties in physical units (um, s, etc.) and not only in pixels and time bins as before,
# and stores the results
if get_extended_analysis:
    sta_data_analysed = analysis.extend_sta_analysis_to_physical_units(
        sta_data_analysed,
        pixels_per_check=nb_pixels_per_check,
        pxl_size_dmd_um=params.pxl_size_dmd,
        sta_frequency=stimulus_frequency,
        directory=check_directory,
        data_filename="sta_data_analysed_extended.pkl",
    )

if show_all_stas:
    # n_sigma: RF ellipse / SNR mask taken at this many standard deviations of the fitted Gaussian
    analysis.plot_all_stas(
        sta_data_analysed, n_sigma=2, order_by_property="rf_snr", border_width=3
    )

## Visualize receptive fields 

- Check the figure saved in the Analysis/ folder.
- Execution time: 30 secs

In [ ]:
%%time

# Re-loading analysed STA data if already computed and saved,
# to run following analysis without re-computing it
# - False if you run the previous cell to compute the analysed STA data, so it is already in memory
# - True if you just want to run following analysis and the analysed STA data has already been computed and saved (in this case only import and first cell are needed)
load_sta_data = True
if load_sta_data:
    sta_data_analysed_file = os.path.join(check_directory, "sta_data_analysed.pkl")
    sta_data_extended_file = os.path.join(
        check_directory, "sta_data_analysed_extended.pkl"
    )
    if os.path.exists(sta_data_extended_file):
        sta_data_analysed = utils.load_obj(sta_data_extended_file)
        print(f"Loaded extended analysed STA data from {sta_data_extended_file}")
    elif os.path.exists(sta_data_analysed_file):
        sta_data_analysed = utils.load_obj(sta_data_analysed_file)
        print(f"Loaded analysed STA data from {sta_data_analysed_file}")
    else:
        print(
            "Analysed STA data file not found. Check the filename or run the previous cell to compute and save the analysed STA data before running this cell."
        )
    # SWN has no repeated sequence -> no rasters to overlay
    rep_seq_data = None
    rep_seq_data_file = os.path.join(check_directory, "Check_rasters_data.pkl")
    if is_swn:
        print("SWN: no repeated-sequence rasters to overlay on the STAs.")
    elif os.path.exists(rep_seq_data_file):
        rep_seq_data = utils.load_obj(rep_seq_data_file)
    else:
        print(
            f"Repeated sequence data file not found at {rep_seq_data_file}. Check the filename or run the previous cell to compute and save the repeated sequence data before running this cell."
        )

# Plotting STAs & their fitted ellipses for all cells
analysis.plot_sta_fitted_with_ellipse(
    sta_data_analysed,
    check_directory,
    add_raster_plot=rep_seq_data is not None,  # SWN has no rasters
    rep_seq_data=rep_seq_data,
    add_spatial_mask=True,
    cell_ids=None,  # list of cell ids to plot, e.g., [208, 209, 210], if None it will plot all cells
    show_figures=False,
    n_sigma=2,  # RF ellipse / SNR mask taken at this many standard deviations of the fitted Gaussian
    xdim=8,
    ydim=5,
    save_format="png",
)

## (Optional) Read the STA results and plot one cell's receptive field

The receptive-field (RF) analysis is saved to **`sta_data_analysed_extended.pkl`** in the
checkerboard analysis folder (`check_directory`). It is a dictionary keyed by cell id:

    sta_data[cell_id]                   # the per-cell dictionary
    sta_data[cell_id]["sta_analysis"]   # <- all the RF results live here

Inside `"sta_analysis"`, the entries you need to describe / plot a RF are:

| key | what it is |
|-----|------------|
| `"Spatial"` | 2D array — the spatial STA (the RF image), in checker pixels |
| `"Temporal"` | 1D array — the temporal STA (time course at the RF centre) |
| `"EllipseCoor"` | `[amp, x0, y0, sigma_x, sigma_y, angle]` of the fitted RF ellipse, in **checker pixels** |
| `"FittedEllipse"` | `True` if the ellipse fit succeeded (`False` → placeholder values) |
| `"Cell_delay"` | frame (time bin) of the spatial STA (the RF peak lag) |
| `"EllipseCoor_um"` | the same ellipse coordinates, converted to **micrometres** |
| `"Spatial_unit_size_um"` | size of one spatial-STA pixel, in µm |
| `"TemporalTimeVector_s"` | time axis of the temporal STA, in seconds |
| `"Cell_delay_s"` | the RF peak lag, in seconds |

The cell below opens the file, picks one example cell, prints these values, and plots the
spatial RF (with its fitted ellipse) next to the temporal STA. It only needs **Cell 1**
(load data) to have been run, so `check_directory` is defined.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Open the saved STA analysis (a dictionary {cell_id: {...}}) ---
sta_file = os.path.join(check_directory, "sta_data_analysed_extended.pkl")
sta_data = utils.load_obj(sta_file)
print(f"{len(sta_data)} cells saved in:\n  {sta_file}\n")

# --- 2. Pick one example cell (change this to any id printed above) ---
example_cell = list(sta_data.keys())[0]
rf = sta_data[example_cell][
    "sta_analysis"
]  # <- all RF info is under the "sta_analysis" key

# --- 3. Extract the relevant entries ---
spatial = rf["Spatial"]  # 2D RF image (checker pixels)
temporal = rf["Temporal"]  # temporal STA (time course)
ellipse = rf["EllipseCoor"]  # [amp, x0, y0, sigma_x, sigma_y, angle], in pixels
ellipse_um = rf["EllipseCoor_um"]  # same, in micrometres
fitted = rf["FittedEllipse"]  # was the ellipse fit successful?

amp, x0, y0, sigma_x, sigma_y, angle = ellipse
print(f"Example cell {example_cell}:")
print(f"  FittedEllipse  : {fitted}")
print(f"  RF centre  (px): x0={x0:.1f}, y0={y0:.1f}")
print(
    f"  RF sigmas  (px): sigma_x={sigma_x:.2f}, sigma_y={sigma_y:.2f}, angle={angle:.0f} deg"
)
if fitted:
    n_sigma = 2
    level_factor = np.exp(-(n_sigma**2) / 2)  # peak fraction of the Gaussian at n_sigma
    diameter_um = utils.ellipse_diameter(ellipse_um, method="circle_approx")
    area_um2 = utils.ellipse_area(ellipse_um, method="formula")
    snr = utils.rf_snr(spatial, ellipse, method="peak_std", level_factor=level_factor)
    print(f"  RF diameter    : {diameter_um:.0f} um   (area {area_um2:.0f} um^2)")
    print(f"  RF SNR         : {snr:.1f}")

# --- 4. Plot: spatial RF (+ fitted ellipse) and temporal STA ---
fig, (ax_rf, ax_t) = plt.subplots(1, 2, figsize=(11, 5))

utils.plot_sta(
    ax_rf, spatial, ellipse
)  # draws the STA image + the fitted-ellipse contour
ax_rf.set_title(f"Cell {example_cell} - spatial receptive field")
ax_rf.set_xticks([])
ax_rf.set_yticks([])

t_axis = rf.get("TemporalTimeVector_s", np.arange(len(temporal)))
ax_t.plot(t_axis, temporal, "k", lw=2)
ax_t.axhline(0, color="k", lw=0.5)
ax_t.set_title("Temporal STA")
ax_t.set_xlabel("Time (s)")
plt.tight_layout()
plt.show()